In [3]:
!pip install openai-whisper python-docx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 4.7 MB/s  0:00:00eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 3.9 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 4.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 MB 5.5 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 4.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.6/797.6 kB 5.0 MB/s  0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=142545936cbf506b2129498b4fa38bc6a75a5a9a71d9cc4e6a37dd999beddf43
  Stored in directory: /home/ksenia/.cache/pip/wheels/be/22/86/dcf050d50e02b3bb67b5a3b289be2f3376ce7f8717d012d6a8
Successfully built openai-whisper
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import whisper
from docx import Document
from docx.shared import Pt
import os

In [6]:
def transcribe_audio(audio_path: str, model_size: str = "medium"):
    """
    Транскрибирует аудио с таймкодами.
    model_size: tiny, base, small, medium, large — чем больше, тем точнее и медленнее.
    """
    print(f"Загружаю модель '{model_size}'...")
    model = whisper.load_model(model_size)

    print("Распознаю речь...")
    result = model.transcribe(
        audio_path,
        language="ru",          # поставь None для автоопределения
        word_timestamps=True,
        verbose=False,
    )
    return result


def group_into_turns(segments, max_gap: float = 1.5):
    """
    Грубая эвристика: если пауза между сегментами > max_gap секунд,
    считаем что сменился говорящий (новая "роль").
    """
    turns = []
    current_text = []
    current_start = None
    last_end = None

    for seg in segments:
        start, end, text = seg["start"], seg["end"], seg["text"].strip()
        if not text:
            continue

        if current_start is None:
            current_start = start
            current_text.append(text)
        elif last_end is not None and (start - last_end) > max_gap:
            # новая реплика
            turns.append({
                "start": current_start,
                "end": last_end,
                "text": " ".join(current_text),
            })
            current_start = start
            current_text = [text]
        else:
            current_text.append(text)

        last_end = end

    if current_text:
        turns.append({
            "start": current_start,
            "end": last_end,
            "text": " ".join(current_text),
        })

    return turns


def fmt_time(seconds: float) -> str:
    m = int(seconds // 60)
    s = int(seconds % 60)
    return f"{m:02d}:{s:02d}"


def save_to_word(turns, output_path: str, assign_speakers: bool = True):
    """
    Сохраняет реплики в Word.
    Если assign_speakers=True — назначает роли Голос 1, Голос 2 и т.д. по кругу.
    """
    doc = Document()
    doc.add_heading("Транскрипция аудио", level=1)

    # Простейшее присвоение ролей по кругу (A, B, A, B...)
    # Для настоящего diarization нужен pyannote.audio — см. ниже.
    speakers = ["Голос 1", "Голос 2"]
    speaker_idx = 0

    for i, turn in enumerate(turns):
        if assign_speakers:
            speaker = speakers[speaker_idx % len(speakers)]
            speaker_idx += 1
        else:
            speaker = "Спикер"

        p = doc.add_paragraph()
        run = p.add_run(f"[{fmt_time(turn['start'])} – {fmt_time(turn['end'])}] {speaker}: ")
        run.bold = True

        p.add_run(turn["text"])

    doc.save(output_path)
    print(f"Готово! Файл сохранён: {output_path}")


def main(audio_file: str, output_docx: str = "transcript.docx"):
    if not os.path.exists(audio_file):
        raise FileNotFoundError(audio_file)

    result = transcribe_audio(audio_file, model_size="medium")
    turns = group_into_turns(result["segments"], max_gap=1.5)
    save_to_word(turns, output_docx)


if __name__ == "__main__":
    main("Новая запись 249.m4a", "transcript.docx")

Загружаю модель 'medium'...


100%|█████████████████████████████████████| 1.42G/1.42G [03:52<00:00, 6.56MiB/s]


Распознаю речь...


FileNotFoundError: [Errno 2] No such file or directory: 'ffmpeg'